# Майский скрипт эквайринга (без Excel)

## Что делает скрипт
- Считает майские показатели витрины **только из Озера**.
- Применяет коэффициент для `commission_monthly` из заранее подготовленного CSV за апрель.
- Формирует новый столбец `com_forcast` и пересчитывает производные метрики.

## Что нужно заполнить перед запуском
В ячейке параметров:
- `report_month = '2026-05-01'`
- `coef_csv_path` — путь до CSV коэффициентов за апрель
  (по умолчанию: `./кэфы_апрель.csv`)
- `output_csv_path` — путь сохранения майской витрины
- `run_invalidate_metadata`, `save_csv`

Важно для коллеги:
- Файл коэффициентов можно положить в любую удобную папку.
- В `coef_csv_path` нужно вручную прописать путь к этому файлу (относительный или абсолютный).

## Как запускать
1. Проверить параметры.
2. Запустить ячейки сверху вниз.
3. Проверить итоговые контрольные выводы.
4. Забрать итоговый CSV майской витрины.

## Важные правила
- Периметр договоров: SA + активные terms (`cf_ter_type='P'`), без `RSHB` в периметре.
- Транзакции: фильтр `RSHB` только в trx-шаге.
- `AUR = retl_cnt * 1926`.
- `commission_monthly` в финальном датасете корректируется через коэффициент из апреля.
- `com_forcast = commission_monthly_lake * k_comm_monthly_agr`.
- Если `agr_id` отсутствует в CSV коэффициентов, используется медианный коэффициент из файла.
- `coef_source`: `agr_ratio` (коэффициент найден) или `fallback` (подставлена медиана).

In [ ]:
import pandas as pd

attributes_dict_df = pd.DataFrame([
    {'attribute': 'report_month', 'description_ru': 'Отчетный месяц витрины (первый день месяца)'},
    {'attribute': 'snapshot_month_start', 'description_ru': 'Техническая метка среза для BI-фильтрации'},
    {'attribute': 'inn', 'description_ru': 'ИНН клиента'},
    {'attribute': 'company_name', 'description_ru': 'Наименование клиента'},
    {'attribute': 'agr_id', 'description_ru': 'ID договора из Alpha (если есть)'},
    {'attribute': 'n_agr', 'description_ru': 'Внутренний номер договора в Alpha'},
    {'attribute': 'contract_number', 'description_ru': 'Номер договора в Alpha'},
    {'attribute': 'd_valid_from', 'description_ru': 'Дата начала действия договора'},
    {'attribute': 'd_valid_to', 'description_ru': 'Дата окончания действия договора'},
    {'attribute': 'cdi_id', 'description_ru': 'Идентификатор клиента в CDI'},
    {'attribute': 'ssp_ocrm', 'description_ru': 'Сегмент/блок ОКРМ'},
    {'attribute': 'ogrn', 'description_ru': 'ОГРН клиента'},
    {'attribute': 'filial_rf', 'description_ru': 'Филиал РФ'},
    {'attribute': 'vsp_name', 'description_ru': 'Наименование ВСП'},
    {'attribute': 'vsp_code', 'description_ru': 'Код ВСП'},
    {'attribute': 'tariff_name', 'description_ru': 'Тарифный план'},
    {'attribute': 'retl_cnt', 'description_ru': 'Количество торговых точек'},
    {'attribute': 'term_cnt', 'description_ru': 'Количество терминалов'},
    {'attribute': 'active_term_cnt', 'description_ru': 'Количество активных терминалов (>=1 trx с суммой > 1 за месяц)'},
    {'attribute': 'trx_cnt', 'description_ru': 'Количество операций за месяц'},
    {'attribute': 'trx_sum', 'description_ru': 'Сумма операций за месяц'},
    {'attribute': 'commission_from_ops', 'description_ru': 'Комиссия с операций'},
    {'attribute': 'commission_monthly', 'description_ru': 'Фиксированная комиссия в месяц'},
    {'attribute': 'int_component', 'description_ru': 'Interchange / внутренняя составляющая'},
    {'attribute': 'commission_total', 'description_ru': 'Итоговая комиссия'},
    {'attribute': 'aur', 'description_ru': 'АУР (расчет по терминалам)'},
    {'attribute': 'amortization', 'description_ru': 'Амортизация'},
    {'attribute': 'chod', 'description_ru': 'ЧОД'},
    {'attribute': 'fin_result', 'description_ru': 'Финансовый результат'},
])

# Справочник оставлен для документации, вывод отключен для компактного запуска.
print(f'Справочник атрибутов загружен: {len(attributes_dict_df)} полей')

In [ ]:
import re
from decimal import Decimal, InvalidOperation
from pathlib import Path

import pandas as pd
from rail_connectors.connection import connect

# Убираем scientific notation в выводе DataFrame (например 1.6543e+08)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

In [ ]:
# Единственный параметр месяца (первый день месяца)
report_month = '2026-05-01'

# Путь для итогового CSV
output_csv_path = './datamart_month_acquiring_2026_05_may_only.csv'

# Путь до коэффициентов commission_monthly, рассчитанных на апреле
coef_csv_path = './кэфы_апрель.csv'

# Целевая доля agr_id, которые должны приходить напрямую из SA (без fallback)
target_agr_id_direct_pct = 99.0

# Управление служебными шагами
run_invalidate_metadata = False
save_csv = True

report_month_ts = pd.to_datetime(report_month)
month_start = report_month_ts.strftime('%Y-%m-%d')
month_end = (report_month_ts + pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')
report_month_label = report_month_ts.strftime('%Y-%m')
snapshot_month_start = month_start

invalidate_tables = [
    'ods_alpha.scd1_agreements',
    'ods_alpha.scd1_companies',
    'ods_alpha.scd1_agr_terms',
    'ocrm_ul.s_org_ext',
    'cdiul.ext_id_org',
    'ods_alpha.scd1_merchants',
    'ods_alpha.scd1_pos_terminals',
    'sandbox_ai.shestopalov_terminal_amortization_oneoff',
    'ods_alpha.scd1_trx',
    'ods_alpha.scd1_trx_acq',
    'ods_alpha.scd1_trx_int',
    'ods_alpha.scd1_base24_fiids',
    'ods.scd1_z_r2_ip_merchants',
    'ods.scd1_z_r2_tariff_tune',
    'ods.scd1_z_r2_tariff_fix',
    'ods.scd1_z_cl_corp',
    'ods.scd1_z_depart',
    'ods.scd1_z_branch',
    'ods.scd1_z_r2_tariff_plan'
]

if run_invalidate_metadata:
    invalidate_ok = 0
    invalidate_failed = []
    with imp:
        for t in invalidate_tables:
            try:
                imp.execute(f'invalidate metadata {t}')
                invalidate_ok += 1
            except Exception as e:
                invalidate_failed.append((t, str(e)))

    print(f'Invalidate успешно: {invalidate_ok}/{len(invalidate_tables)}')
    if invalidate_failed:
        print(f'Invalidate с ошибкой: {len(invalidate_failed)} (продолжаем выполнение)')
        display(pd.DataFrame(invalidate_failed, columns=['table_name', 'error_message']).head(20))
else:
    print('Invalidate metadata пропущен (run_invalidate_metadata=False) для ускорения запуска.')

In [ ]:
def normalize_inn(v):
    if pd.isna(v):
        return None
    s = re.sub(r'[^0-9]', '', str(v).strip())
    return s or None


def normalize_contract(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    return s if s else None


def to_decimal_or_none(v):
    if pd.isna(v):
        return None
    if isinstance(v, Decimal):
        return v
    try:
        return Decimal(str(v).strip().replace(',', '.'))
    except (InvalidOperation, ValueError):
        return None

## SA-периметр (изменен)

Берем SA-договоры, активные в выбранном месяце, и оставляем только те, у которых существует активная запись в `ods_alpha.scd1_agr_terms` с `cf_ter_type='P'`.

Важно: здесь **нет** фильтра по `c_fiid_grp='RSHB'`.

In [ ]:
sql_sa_perimeter = f"""
select distinct
  cast(a.n_agr as string) as n_agr,
  cast(a.abs_agr_id as string) as agr_id,
  cast(a.n_cmp_client as string) as n_cmp_client,
  cast(a.c_agr_number as string) as contract_number,
  cast(a.d_valid_from as date) as d_valid_from,
  cast(a.d_valid_to as date) as d_valid_to,
  regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn,
  cast(c.c_cmp_name as string) as company_name
from ods_alpha.scd1_agreements a
join ods_alpha.scd1_companies c
  on c.n_cmp = a.n_cmp_client
where upper(trim(cast(a.acq_class as string))) = 'SA'
  and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
  and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
  and coalesce(a.ods_deleted_flg, '0') <> '1'
  and coalesce(c.ods_deleted_flg, '0') <> '1'
  and c.c_inn is not null
  and exists (
      select 1
      from ods_alpha.scd1_agr_terms t
      where cast(t.n_agr as string) = cast(a.n_agr as string)
        and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
        and (t.d_valid_to is null or cast(t.d_valid_to as date) > cast('{month_start}' as date))
        and upper(trim(cast(t.cf_ter_type as string))) = 'P'
        and coalesce(t.ods_deleted_flg, '0') <> '1'
  )
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    sa_df = imp.fetch(sql_sa_perimeter)

if sa_df is None:
    sa_df = pd.DataFrame()

if not sa_df.empty:
    sa_df['inn'] = sa_df['inn'].map(normalize_inn)
    sa_df['contract_number'] = sa_df['contract_number'].map(normalize_contract)

sa_rows_cnt = len(sa_df)
sa_inn_unique_cnt = sa_df['inn'].dropna().astype(str).nunique() if 'inn' in sa_df.columns else 0
print(f'SA-периметр: строк={sa_rows_cnt:,}, уникальных ИНН={sa_inn_unique_cnt:,}')

In [ ]:
inn_values = sorted([x for x in sa_df.get('inn', pd.Series(dtype=object)).dropna().astype(str).unique().tolist() if x])
inn_sql_list = ', '.join([f"'{x}'" for x in inn_values]) if inn_values else "''"

sql_cdi = f"""
with ocrm_current as (
  select
    regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '') as inn,
    cast(soe.row_id as string) as row_id,
    trim(cast(soe.x_area_resp as string)) as x_area_resp_norm,
    case
      when soe.x_area_resp like 'ДММБ%' then 'ДМ'
      when soe.x_area_resp like 'ДМСБ (ми%' then 'ДМ'
      when soe.x_area_resp like 'ДМСБ(ми%' then 'ДМ'
      when soe.x_area_resp like 'ДМСБ (ма%' then 'ДМСБ'
      when soe.x_area_resp like 'ДМСБ(ма%' then 'ДМСБ'
      when soe.x_area_resp like 'ДМСБ (ср%' then 'ДМСБ'
      when soe.x_area_resp like 'ДМСБ(ср%' then 'ДМСБ'
      when soe.x_area_resp like 'ДСБ%' then 'ДМСБ'
      when soe.x_area_resp like 'ДКБ%' then 'ДКБ'
      when soe.x_area_resp like 'ДРПА%' then 'ДРПА'
      when soe.x_area_resp like 'ДРА%' then 'ДРПА'
      when lower(soe.x_area_resp) like 'не под%' then 'Не подлежит сегментации'
      when nullif(trim(cast(soe.x_area_resp as string)), '') is null then null
      else null
    end as ssp_ocrm,
    row_number() over (
      partition by regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '')
      order by cast(soe.created as timestamp) desc,
               cast(soe.row_id as string) desc
    ) as rn
  from ocrm_ul.s_org_ext soe
  where regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '') in ({inn_sql_list})
    and coalesce(soe.x_removed_flg, 'N') = 'N'
    and coalesce(soe.x_duplicate_flg, 'N') = 'N'
),
ocrm_one as (
  select inn, row_id, ssp_ocrm, x_area_resp_norm
  from ocrm_current
  where rn = 1
)
select
  o.inn,
  o.ssp_ocrm,
  o.x_area_resp_norm,
  cast(e.party_id as string) as cdi_id
from ocrm_one o
left join cdiul.ext_id_org e
  on cast(e.cmo_ext_party_source_id as string) = o.row_id
 and upper(cast(e.cmo_ext_source_system as string)) like 'OCRM%'
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    cdi_map_df = imp.fetch(sql_cdi)

if cdi_map_df is None:
    cdi_map_df = pd.DataFrame(columns=['inn', 'ssp_ocrm', 'x_area_resp_norm', 'cdi_id'])

if not cdi_map_df.empty:
    cdi_map_df['inn'] = cdi_map_df['inn'].map(normalize_inn)
    cdi_map_df['cdi_id'] = cdi_map_df['cdi_id'].astype(str)
    allowed_ssp_values = {'ДМ', 'ДМСБ', 'ДКБ', 'ДРПА', 'Не подлежит сегментации'}
    cdi_map_df['ssp_ocrm'] = cdi_map_df['ssp_ocrm'].where(cdi_map_df['ssp_ocrm'].isin(allowed_ssp_values), None)
    cdi_map_df = cdi_map_df.drop_duplicates(subset=['inn'], keep='first')

print(f'CDI map: строк={len(cdi_map_df):,}')

In [ ]:
cdi_values = sorted([x for x in cdi_map_df.get('cdi_id', pd.Series(dtype=object)).dropna().astype(str).unique().tolist() if x])
cdi_sql_list = ', '.join([f"'{x}'" for x in cdi_values]) if cdi_values else "''"

sql_cft = f"""
select
  cast(e.party_id as string) as cdi_id,
  cast(e.cmo_ext_party_source_id as string) as cft_id
from cdiul.ext_id_org e
where cast(e.party_id as string) in ({cdi_sql_list})
  and upper(cast(e.cmo_ext_source_system as string)) like 'CFT%'
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    cft_map_df = imp.fetch(sql_cft)

if cft_map_df is None:
    cft_map_df = pd.DataFrame(columns=['cdi_id', 'cft_id'])

if not cft_map_df.empty:
    cft_map_df['cdi_id'] = cft_map_df['cdi_id'].astype(str)
    cft_map_df['cft_id'] = cft_map_df['cft_id'].astype(str)
    cft_map_df = cft_map_df.drop_duplicates(subset=['cdi_id'], keep='first')

print(f'CFT map: строк={len(cft_map_df):,}')

In [ ]:
# 1) Agreement-level metrics with active terminals in month (оптимизировано)
if sa_df.empty:
    cmp_df = pd.DataFrame(columns=['n_agr', 'n_cmp_client', 'retl_cnt', 'term_cnt', 'amortization'])
    trx_df = pd.DataFrame(columns=['n_agr', 'trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component', 'n_cmp_client', 'active_term_cnt'])
else:
    n_agrs_cmp = sorted(sa_df.get('n_agr', pd.Series(dtype=object)).dropna().astype(str).unique().tolist())
    agr_in_cmp = ', '.join([f"'{x}'" for x in n_agrs_cmp]) if n_agrs_cmp else "''"

    sql_cmp = f"""
    with sa_agr as (
      select distinct
        cast(a.n_agr as string) as n_agr,
        cast(a.n_cmp_client as string) as n_cmp_client
      from ods_alpha.scd1_agreements a
      where cast(a.n_agr as string) in ({agr_in_cmp})
    ),
    m as (
      select
        sa.n_agr,
        sa.n_cmp_client,
        cast(mm.c_nmrc as string) as c_nmrc
      from sa_agr sa
      join ods_alpha.scd1_merchants mm
        on cast(mm.n_cmp as string) = sa.n_cmp_client
      where mm.c_nmrc is not null
        and coalesce(mm.ods_deleted_flg, '0') <> '1'
      group by sa.n_agr, sa.n_cmp_client, cast(mm.c_nmrc as string)
    ),
    term_active as (
      select
        m.n_agr,
        m.n_cmp_client,
        m.c_nmrc,
        cast(t.c_nter as string) as c_nter
      from m
      join ods_alpha.scd1_pos_terminals t
        on cast(t.c_nmrc as string) = m.c_nmrc
      where t.c_nter is not null
        and coalesce(t.ods_deleted_flg, '0') <> '1'
        and coalesce(cast(t.d_ter_install as date), cast('1900-01-01' as date)) <= cast('{month_end}' as date)
        and coalesce(cast(t.d_ter_close as date), cast('2999-12-31' as date)) >= cast('{month_start}' as date)
      group by m.n_agr, m.n_cmp_client, m.c_nmrc, cast(t.c_nter as string)
    ),
    retl as (
      select n_agr, count(distinct c_nmrc) as retl_cnt
      from term_active
      group by n_agr
    ),
    term as (
      select n_agr, count(distinct c_nter) as term_cnt
      from term_active
      group by n_agr
    ),
    amort as (
      select
        ta.n_agr,
        sum(coalesce(cast(am.amortization_monthly as double), 0.0)) as amortization
      from term_active ta
      left join sandbox_ai.shestopalov_terminal_amortization_oneoff am
        on cast(am.c_nter as string) = ta.c_nter
      group by ta.n_agr
    )
    select
      sa.n_agr,
      sa.n_cmp_client,
      r.retl_cnt,
      t.term_cnt,
      a.amortization
    from sa_agr sa
    left join retl r on r.n_agr = sa.n_agr
    left join term t on t.n_agr = sa.n_agr
    left join amort a on a.n_agr = sa.n_agr
    """

    with imp:
        imp.execute('set MEM_LIMIT=12g')
        cmp_df = imp.fetch(sql_cmp)

    if cmp_df is None:
        cmp_df = pd.DataFrame(columns=['n_agr', 'n_cmp_client', 'retl_cnt', 'term_cnt', 'amortization'])

    if not cmp_df.empty:
        cmp_df['n_agr'] = cmp_df['n_agr'].astype(str)
        cmp_df['n_cmp_client'] = cmp_df['n_cmp_client'].astype(str)

    # 2) TRX metrics by agreement (RSHB в trx оставлен как в базовой тетрадке)
    n_agrs = n_agrs_cmp
    agr_in = ', '.join([f"'{x}'" for x in n_agrs]) if n_agrs else "''"

    sql_trx = f"""
    with sa_agr as (
      select distinct cast(n_agr as string) as n_agr, cast(n_cmp_client as string) as n_cmp_client
      from ods_alpha.scd1_agreements
      where cast(n_agr as string) in ({agr_in})
    ),
    trx_base_raw as (
      select
        cast(t.n_trx as string) as n_trx,
        cast(t.c_nter as string) as c_nter,
        cast(t.n_amt_src as double) as n_amt_src
      from ods_alpha.scd1_trx t
      left join ods_alpha.scd1_base24_fiids fa
        on cast(fa.c_fiid as string) = cast(t.c_fiid_acq as string)
      where to_date(cast(t.d_trx_orig as timestamp)) between cast('{month_start}' as date) and cast('{month_end}' as date)
        and t.c_nter is not null
        and coalesce(t.ods_deleted_flg, '0') <> '1'
        and t.c_trx_class = 'SA'
        and t.c_trx_type = 'S01'
        and coalesce(t.cf_trx_stat, '') <> 'R'
        and coalesce(cast(fa.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
    ),
    trx_base as (
      select
        n_trx,
        max(c_nter) as c_nter,
        max(n_amt_src) as n_amt_src
      from trx_base_raw
      group by n_trx
    ),
    ta_raw as (
      select
        cast(a.n_trx as string) as n_trx,
        cast(a.n_agr as string) as n_agr,
        coalesce(cast(a.n_amt_tax as double), 0.0) as n_amt_tax
      from ods_alpha.scd1_trx_acq a
      join trx_base tb
        on tb.n_trx = cast(a.n_trx as string)
      where cast(a.n_agr as string) in ({agr_in})
    ),
    ta as (
      select
        n_trx,
        n_agr,
        max(n_amt_tax) as n_amt_tax
      from ta_raw
      group by n_trx, n_agr
    ),
    trx_keys as (
      select distinct n_trx
      from ta
    ),
    trx_int_agg as (
      select
        cast(i.n_trx as string) as n_trx,
        sum(coalesce(cast(i.n_amt_fee as double), 0.0)) as n_amt_fee
      from ods_alpha.scd1_trx_int i
      join trx_keys k
        on k.n_trx = cast(i.n_trx as string)
      group by cast(i.n_trx as string)
    ),
    tj as (
      select ta.n_agr, sa.n_cmp_client, ta.n_trx, tb.c_nter, tb.n_amt_src, ta.n_amt_tax
      from ta
      join trx_base tb
        on tb.n_trx = ta.n_trx
      left join sa_agr sa
        on sa.n_agr = ta.n_agr
    ),
    trx_agg as (
      select
        tj.n_agr,
        count(distinct tj.n_trx) as trx_cnt,
        sum(tj.n_amt_src) as trx_sum,
        sum(tj.n_amt_tax) as commission_from_ops,
        sum(coalesce(i.n_amt_fee, 0.0)) as int_component
      from tj
      left join trx_int_agg i
        on i.n_trx = tj.n_trx
      group by tj.n_agr
    ),
    active_term_agg as (
      select
        tj.n_cmp_client,
        count(distinct case when coalesce(tj.n_amt_src, 0.0) > 1 then tj.c_nter else null end) as active_term_cnt
      from tj
      where tj.n_cmp_client is not null
      group by tj.n_cmp_client
    )
    select
      t.n_agr,
      t.trx_cnt,
      t.trx_sum,
      t.commission_from_ops,
      t.int_component,
      a.n_cmp_client,
      a.active_term_cnt
    from trx_agg t
    left join sa_agr sa
      on sa.n_agr = t.n_agr
    left join active_term_agg a
      on a.n_cmp_client = sa.n_cmp_client
    """

    with imp:
        imp.execute('set MEM_LIMIT=16g')
        trx_df = imp.fetch(sql_trx)

    if trx_df is None:
        trx_df = pd.DataFrame(columns=['n_agr', 'trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component', 'n_cmp_client', 'active_term_cnt'])

    if not trx_df.empty:
        trx_df['n_agr'] = trx_df['n_agr'].astype(str)
        trx_df['n_cmp_client'] = trx_df['n_cmp_client'].astype(str)

active_term_df = (
    trx_df[['n_cmp_client', 'active_term_cnt']]
    .dropna(subset=['n_cmp_client'])
    .drop_duplicates(subset=['n_cmp_client'])
    if len(trx_df)
    else pd.DataFrame(columns=['n_cmp_client', 'active_term_cnt'])
)

# 3) R2 attributes вынесены в следующую секцию

# 4) Final assembly вынесена в отдельную секцию

## Секция 4. R2 атрибуты
Кратко: получаем оргструктуру и базовую фикс-комиссию по `cft_id` для обогащения витрины.

In [ ]:
# R2 атрибуты: оргструктура, тариф и фикс-комиссия
cft_values = sorted([x for x in cft_map_df.get('cft_id', pd.Series(dtype=object)).dropna().astype(str).unique().tolist() if x])
cft_sql_list = ', '.join([f"'{x}'" for x in cft_values]) if cft_values else "''"

if not cft_values:
    r2_df = pd.DataFrame(columns=['r2_id', 'cft_id', 'ogrn', 'vsp_name', 'vsp_code', 'filial_rf', 'tariff_name', 'commission_monthly'])
else:
    sql_r2 = f"""
    with r2 as (
      select
        cast(m.id as string) as r2_id,
        cast(m.c_cl_org as string) as cft_id,
        cast(m.c_depart as string) as c_depart,
        cast(m.c_tariff_plan as string) as c_tariff_plan
      from ods.scd1_z_r2_ip_merchants m
      where cast(m.c_cl_org as string) in ({cft_sql_list})
    ),
    fix as (
      select cast(tt.c_tariff_plan as string) as c_tariff_plan, max(cast(tf.c_summa as decimal(18,2))) as commission_monthly_fix
      from ods.scd1_z_r2_tariff_tune tt
      left join ods.scd1_z_r2_tariff_fix tf on tt.c_tariff = tf.id
      group by cast(tt.c_tariff_plan as string)
    )
    select
      r2.r2_id,
      r2.cft_id,
      cast(corp.c_register_gos_reg_num_rec as string) as ogrn,
      cast(dep.c_name as string) as vsp_name,
      cast(dep.c_num as string) as vsp_code,
      case
        when br.c_shortlabel is null then null
        when upper(cast(br.c_shortlabel as string)) like '%РФ%'
          then regexp_extract(cast(br.c_shortlabel as string), '^(.*?РФ)', 1)
        else cast(br.c_shortlabel as string)
      end as filial_rf,
      cast(tp.c_name as string) as tariff_name,
      cast(fix.commission_monthly_fix as decimal(18,2)) as commission_monthly
    from r2
    left join ods.scd1_z_cl_corp corp on cast(corp.id as string) = r2.cft_id
    left join ods.scd1_z_depart dep on cast(dep.id as string) = r2.c_depart
    left join ods.scd1_z_branch br on cast(br.id as string) = cast(dep.c_filial as string)
    left join ods.scd1_z_r2_tariff_plan tp on cast(tp.id as string) = r2.c_tariff_plan
    left join fix on fix.c_tariff_plan = r2.c_tariff_plan
    """

    with imp:
        imp.execute('set MEM_LIMIT=8g')
        r2_df = imp.fetch(sql_r2)

    if r2_df is None:
        r2_df = pd.DataFrame(columns=['r2_id', 'cft_id', 'ogrn', 'vsp_name', 'vsp_code', 'filial_rf', 'tariff_name', 'commission_monthly'])

if not r2_df.empty:
    for c in ['r2_id', 'cft_id', 'ogrn', 'vsp_name', 'vsp_code', 'filial_rf', 'tariff_name']:
        r2_df[c] = r2_df[c].astype(str)
    r2_df = r2_df.drop_duplicates(subset=['cft_id'], keep='first')

print(f'R2 rows: {len(r2_df):,}')

## Секция 5. Сборка финальной витрины
Кратко: объединяем SA, CDI/CFT, операционные и транзакционные метрики, считаем финальные показатели и формируем `final_df`.

In [ ]:
# Final assembly + fallback по agr_id + финальные формулы
base_df = sa_df.copy()

if not cdi_map_df.empty and not base_df.empty:
    base_df = base_df.merge(cdi_map_df[['inn', 'cdi_id', 'ssp_ocrm']], on='inn', how='left')
else:
    base_df['cdi_id'] = None
    base_df['ssp_ocrm'] = None

if not cft_map_df.empty and not base_df.empty:
    base_df = base_df.merge(cft_map_df[['cdi_id', 'cft_id']], on='cdi_id', how='left')
else:
    base_df['cft_id'] = None

if not r2_df.empty and not base_df.empty:
    base_df = base_df.merge(r2_df, on='cft_id', how='left')
else:
    for col in ['r2_id', 'commission_monthly', 'ogrn', 'filial_rf', 'vsp_name', 'vsp_code', 'tariff_name']:
        base_df[col] = None

if not cmp_df.empty and not base_df.empty:
    base_df = base_df.merge(cmp_df[['n_agr', 'retl_cnt', 'term_cnt', 'amortization']], on='n_agr', how='left')
else:
    for col in ['retl_cnt', 'term_cnt', 'amortization']:
        base_df[col] = None

if 'active_term_df' in globals() and not active_term_df.empty and not base_df.empty:
    base_df = base_df.merge(active_term_df[['n_cmp_client', 'active_term_cnt']], on='n_cmp_client', how='left')
else:
    base_df['active_term_cnt'] = None

if not trx_df.empty and not base_df.empty:
    base_df = base_df.merge(trx_df[['n_agr', 'trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component']], on='n_agr', how='left')
else:
    for col in ['trx_cnt', 'trx_sum', 'commission_from_ops', 'int_component']:
        base_df[col] = None

if 'agr_id' not in base_df.columns:
    base_df['agr_id'] = None
if 'r2_id' not in base_df.columns:
    base_df['r2_id'] = None

agr_id_before_fill = base_df['agr_id'].notna() & (base_df['agr_id'].astype(str).str.strip() != '')
base_df['agr_id_source'] = 'sa'
fallback_mask = (~agr_id_before_fill) & base_df['r2_id'].notna() & (base_df['r2_id'].astype(str).str.strip() != '')
base_df.loc[fallback_mask, 'agr_id'] = base_df.loc[fallback_mask, 'r2_id']
base_df.loc[fallback_mask, 'agr_id_source'] = 'r2_fallback'

agr_id_after_fill = base_df['agr_id'].notna() & (base_df['agr_id'].astype(str).str.strip() != '')
missing_after_fill_mask = ~agr_id_after_fill

total_rows = int(len(base_df))
sa_direct_rows = int(agr_id_before_fill.sum())
fallback_rows = int(fallback_mask.sum())
missing_after_rows = int(missing_after_fill_mask.sum())

sa_direct_pct = round(100.0 * sa_direct_rows / total_rows, 2) if total_rows else 0.0
fallback_pct = round(100.0 * fallback_rows / total_rows, 2) if total_rows else 0.0
missing_after_pct = round(100.0 * missing_after_rows / total_rows, 2) if total_rows else 0.0
agr_filled_after_pct = round(100.0 * agr_id_after_fill.mean(), 2) if total_rows else 0.0

agr_source_stats_df = pd.DataFrame([
    {'source': 'sa_direct', 'rows_cnt': sa_direct_rows, 'rows_pct': sa_direct_pct},
    {'source': 'r2_fallback', 'rows_cnt': fallback_rows, 'rows_pct': fallback_pct},
    {'source': 'missing_after_fallback', 'rows_cnt': missing_after_rows, 'rows_pct': missing_after_pct},
])

print(f'Всего строк: {total_rows:,}')
print(f'agr_id из SA (без fallback): {sa_direct_rows:,} ({sa_direct_pct}%)')
print(f'agr_id через fallback (r2_id): {fallback_rows:,} ({fallback_pct}%)')
print(f'agr_id заполнено после fallback: {agr_filled_after_pct}%')
print(f'agr_id отсутствует после fallback: {missing_after_rows:,} ({missing_after_pct}%)')

if sa_direct_pct >= target_agr_id_direct_pct:
    print(f'OK: доля agr_id из SA >= {target_agr_id_direct_pct}%')
else:
    print(f'WARN: доля agr_id из SA ниже целевого порога {target_agr_id_direct_pct}%')

commission_from_ops_num = pd.to_numeric(base_df.get('commission_from_ops'), errors='coerce').fillna(0)
commission_monthly_num = pd.to_numeric(base_df.get('commission_monthly'), errors='coerce').fillna(0)
int_component_num = pd.to_numeric(base_df.get('int_component'), errors='coerce').fillna(0)
amortization_num = pd.to_numeric(base_df.get('amortization'), errors='coerce').fillna(0)

base_df['commission_total'] = commission_from_ops_num + commission_monthly_num

# AUR по бизнес-правилу: только от количества торговых точек
retl_cnt_num = pd.to_numeric(base_df.get('retl_cnt'), errors='coerce').fillna(0)
base_df['aur'] = retl_cnt_num * 1926

aur_num = pd.to_numeric(base_df.get('aur'), errors='coerce').fillna(0)
base_df['chod'] = base_df['commission_total'] + int_component_num
base_df['fin_result'] = base_df['chod'] - aur_num - amortization_num
base_df['report_month'] = report_month_label
base_df['snapshot_month_start'] = snapshot_month_start

mvp_columns = [
    'report_month', 'snapshot_month_start', 'inn', 'company_name',
    'agr_id', 'n_agr', 'contract_number', 'd_valid_from', 'd_valid_to',
    'cdi_id', 'ssp_ocrm', 'ogrn', 'filial_rf', 'vsp_name', 'vsp_code', 'tariff_name',
    'retl_cnt', 'term_cnt', 'active_term_cnt', 'trx_cnt', 'trx_sum',
    'commission_from_ops', 'commission_monthly', 'int_component', 'commission_total',
    'aur', 'amortization', 'chod', 'fin_result'
]

for col in mvp_columns:
    if col not in base_df.columns:
        base_df[col] = None

for col in ['trx_sum', 'commission_from_ops', 'commission_monthly', 'int_component', 'commission_total', 'aur', 'amortization', 'chod', 'fin_result']:
    if col in base_df.columns:
        base_df[col] = base_df[col].map(to_decimal_or_none)

for col in ['retl_cnt', 'term_cnt', 'active_term_cnt', 'trx_cnt']:
    if col in base_df.columns:
        base_df[col] = pd.to_numeric(base_df[col], errors='coerce').astype('Int64')

final_df = base_df[mvp_columns].copy()

print(f'Собрано строк: {len(final_df):,}')
print(f'Атрибутов в финале: {len(final_df.columns)}')

## Секция 6. Финальная витрина до корректировки
Кратко: формируем базовый `final_df` из Озера; финальный CSV сохраняется после применения коэффициентов.

In [ ]:
print('Предварительный экспорт из секции 6 отключен: итоговый экспорт выполняется после применения коэффициентов.')

## Секция 7. Применение апрельских коэффициентов
Кратко: загружаем CSV коэффициентов `commission_monthly` по `agr_id`, применяем к майской витрине и пересчитываем производные метрики.

In [ ]:
# Нормализация ключей для применения коэффициентов по agr_id

def normalize_agr_q1(value):
    if pd.isna(value):
        return None

    s = str(value).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    if s in {'', 'nan', 'None'}:
        return None

    try:
        d = Decimal(s)
        if d == d.to_integral_value():
            return str(int(d))
    except (InvalidOperation, ValueError):
        pass

    s = re.sub(r'\.0$', '', s)
    if s in {'', 'nan', 'None'}:
        return None
    return s

In [ ]:
# Применение коэффициентов commission_monthly (апрель -> май)
coef_df = pd.read_csv(coef_csv_path)

required_coef_cols = {'agr_id_key', 'k_comm_monthly_agr'}
missing_coef_cols = required_coef_cols - set(coef_df.columns)
if missing_coef_cols:
    raise ValueError(f'В файле коэффициентов отсутствуют колонки: {missing_coef_cols}. Доступные: {list(coef_df.columns)}')

coef_df['agr_id_key'] = coef_df['agr_id_key'].apply(normalize_agr_q1)
coef_df['k_comm_monthly_agr'] = pd.to_numeric(coef_df['k_comm_monthly_agr'], errors='coerce')

if 'coef_source' not in coef_df.columns:
    coef_df['coef_source'] = 'agr_ratio'

coef_df = coef_df[['agr_id_key', 'k_comm_monthly_agr', 'coef_source']].drop_duplicates(subset=['agr_id_key'], keep='first')

final_df['agr_id_key'] = final_df['agr_id'].apply(normalize_agr_q1)
final_df['commission_monthly_lake'] = pd.to_numeric(final_df['commission_monthly'], errors='coerce').fillna(0)

final_df = final_df.merge(coef_df, on='agr_id_key', how='left')

median_k = float(coef_df['k_comm_monthly_agr'].dropna().median()) if coef_df['k_comm_monthly_agr'].notna().any() else 1.0
missing_k_mask = final_df['k_comm_monthly_agr'].isna()
final_df.loc[missing_k_mask, 'k_comm_monthly_agr'] = median_k
final_df.loc[missing_k_mask, 'coef_source'] = 'fallback'
final_df['coef_source'] = final_df['coef_source'].fillna('agr_ratio')

# Новый столбец прогноза по commission_monthly
final_df['com_forcast'] = final_df['commission_monthly_lake'] * final_df['k_comm_monthly_agr']

# В финальном commission_monthly используем скорректированное значение
final_df['commission_monthly'] = final_df['com_forcast']

print(f'Коэффициенты загружены: {len(coef_df):,}')
print(f'Median k для fallback: {median_k:,.6f}')
print(f'Строк с fallback коэффициента: {int(missing_k_mask.sum()):,}')

In [ ]:
# Пересчет производных метрик после корректировки commission_monthly
commission_from_ops_num = pd.to_numeric(final_df.get('commission_from_ops'), errors='coerce').fillna(0)
commission_monthly_num = pd.to_numeric(final_df.get('commission_monthly'), errors='coerce').fillna(0)
int_component_num = pd.to_numeric(final_df.get('int_component'), errors='coerce').fillna(0)
amortization_num = pd.to_numeric(final_df.get('amortization'), errors='coerce').fillna(0)
aur_num = pd.to_numeric(final_df.get('aur'), errors='coerce').fillna(0)

final_df['commission_total'] = commission_from_ops_num + commission_monthly_num
final_df['chod'] = final_df['commission_total'] + int_component_num
final_df['fin_result'] = final_df['chod'] - aur_num - amortization_num

print('Контрольные суммы после корректировки commission_monthly:')
print(f"commission_monthly_lake_total: {final_df['commission_monthly_lake'].sum():,.2f}")
print(f"commission_monthly_forecast_total: {final_df['com_forcast'].sum():,.2f}")
print(f"commission_total_total: {pd.to_numeric(final_df['commission_total'], errors='coerce').fillna(0).sum():,.2f}")

In [ ]:
# Экспорт итоговой майской витрины
if save_csv:
    output_path = Path(output_csv_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    export_columns = [
        'report_month', 'snapshot_month_start', 'inn', 'company_name',
        'agr_id', 'n_agr', 'contract_number', 'd_valid_from', 'd_valid_to',
        'cdi_id', 'ssp_ocrm', 'ogrn', 'filial_rf', 'vsp_name', 'vsp_code', 'tariff_name',
        'retl_cnt', 'term_cnt', 'active_term_cnt', 'trx_cnt', 'trx_sum',
        'commission_from_ops', 'commission_monthly_lake', 'k_comm_monthly_agr', 'com_forcast',
        'commission_monthly', 'commission_total', 'int_component', 'amortization', 'aur', 'chod', 'fin_result',
        'coef_source'
    ]
    export_columns = [c for c in export_columns if c in final_df.columns]

    final_df[export_columns].to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f'CSV сохранен: {output_path.resolve()}')
    print(f'Экспортировано колонок: {len(export_columns)}')
else:
    print('Экспорт в CSV отключен (save_csv=False).')

In [ ]:
# Контроль наличия ключевых колонок майского скрипта
required_cols = [
    'commission_monthly_lake', 'k_comm_monthly_agr', 'commission_monthly',
    'com_forcast', 'coef_source'
]
missing_cols = [c for c in required_cols if c not in final_df.columns]
if missing_cols:
    raise ValueError(f'В final_df отсутствуют обязательные колонки: {missing_cols}')

print('Проверка колонок пройдена: все обязательные поля присутствуют.')
print(f"Доля fallback по коэффициентам: {(final_df['coef_source'] == 'fallback').mean() * 100:.2f}%")

final_df[required_cols].head(5)